# Spark 4.0: Imperative Data Pipeline Workshop

## A Traditional Approach to Building Data Pipelines

---

**Version:** Apache Spark 4.0.x (concepts also work with Spark 3.5.x)  
**Approach:** Imperative (traditional PySpark)  
**Companion Notebook:** `spark41_declarative_pipeline.ipynb` (run side-by-side)

---

### Workshop Overview

This notebook teaches you how to build a **production-grade data pipeline** using the traditional imperative approach in Apache Spark 4.0. You will:

1. Understand the business domain and data model
2. Build a medallion architecture (Bronze -> Silver -> Gold)
3. Learn best practices for PySpark transformations
4. Compare this approach with the declarative style in Spark 4.1

### How to Use This Workshop

**If running side-by-side with Spark 4.1 notebook:**
- Open both notebooks in JupyterLab (drag one tab to create split view)
- Execute cells in the same order in both notebooks
- Compare the code structure, verbosity, and approach
- Note the section numbers match between notebooks

### Running This Notebook

**Option 1: Docker Jupyter (Recommended for learning)**
```bash
docker compose -f docker-compose-notebooks.yml up -d
# Open http://localhost:8889
```
This runs Spark in local mode with temp views (no external dependencies).

**Option 2: Full Cluster (Production)**
```bash
# Start the lakehouse stack
./lakehouse start all

# Generate and load test data
./lakehouse testdata generate --days 90
./lakehouse testdata load

# Run via spark-submit
docker exec spark-master /opt/spark/bin/spark-submit \
    --master spark://spark-master:7077 \
    --conf spark.sql.iceberg.vectorization.enabled=false \
    /scripts/pipeline_spark40.py
```

---

## Section 1: The Business Domain

### What is a Ghost Kitchen?

A **ghost kitchen** (also called a virtual kitchen, cloud kitchen, or dark kitchen) is a professional food preparation facility that produces food exclusively for delivery. Unlike traditional restaurants, ghost kitchens have:

- **No dine-in customers** - delivery only
- **Multiple virtual brands** - one physical kitchen operates 5-10 different "restaurants"
- **Optimized for throughput** - designed for high-volume delivery orders
- **Data-driven operations** - every aspect is tracked and optimized

### Our Fictional Company: Casper's Kitchens

In this workshop, we're building a data pipeline for **Casper's Kitchens**, a ghost kitchen operator with:

| Metric | Value |
|--------|-------|
| Virtual Brands | 20 (Burger Republic, Wok This Way, Pizza Planet, etc.) |
| Markets | 4 cities (San Francisco, Silicon Valley, Seattle, Austin) |
| Menu Items | 160 items across 10 food categories |
| Daily Orders | ~835 per day (varies by hour and day of week) |
| Data Volume | ~160K order events per day (~14.5M over 90 days) |

### The Business Questions We're Solving

The data team at Casper's Kitchens needs to answer:

1. **Operations:** What are the average kitchen prep times by brand? Where are bottlenecks?
2. **Delivery:** What's the average delivery time by location? What's the P95?
3. **Revenue:** Which brands generate the most revenue? What's the average order value?
4. **Growth:** Which brands are growing vs declining? How does demand vary by hour?

### Why a Data Pipeline?

Raw operational data is **messy and hard to query**. We need to:

1. **Clean** - Remove nulls, fix data quality issues
2. **Enrich** - Parse nested JSON, add computed fields
3. **Aggregate** - Pre-compute metrics for fast dashboards
4. **Organize** - Structure data in layers for different use cases

---

## Section 2: The Data Model

### Order Lifecycle Events

Every order generates multiple events as it moves through the fulfillment process:

```
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│ order_created   │────▶│ kitchen_started │────▶│ kitchen_finished│
└─────────────────┘     └─────────────────┘     └─────────────────┘
                                                        │
                                                        ▼
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│    delivered    │◀────│driver_picked_up │◀────│   order_ready   │
└─────────────────┘     └─────────────────┘     └─────────────────┘
        ▲                       │
        │               ┌───────┴───────┐
        └───────────────│  driver_ping  │ (every 60 seconds)
                        └───────────────┘
```

### Event Schema

Each event contains:

| Field | Type | Description |
|-------|------|-------------|
| `event_id` | string | Unique identifier for this event |
| `event_type` | string | One of: order_created, kitchen_started, etc. |
| `ts` | string | ISO timestamp of the event |
| `order_id` | string | Links all events for one order |
| `location_id` | int | Which city/kitchen processed this |
| `sequence` | int | Order of events (1, 2, 3...) |
| `body` | string | JSON with event-specific data |

### The `body` JSON Field

The `body` field contains different data depending on event type:

```json
// order_created event
{
  "brand_id": 5,
  "item_ids": "[12, 34, 56]",
  "total": 45.97,
  "lat": 37.7749,
  "lng": -122.4194
}

// driver_ping event
{
  "driver_id": "DRV-1234",
  "lat": 37.7812,
  "lng": -122.4098
}
```

### Dimension Tables

| Table | Records | Description |
|-------|---------|-------------|
| `dim_brands` | 20 | Virtual restaurant brands |
| `dim_categories` | 10 | Food categories (Pizza, Asian, etc.) |
| `dim_items` | 160 | Menu items with prices |
| `dim_locations` | 4 | Delivery markets/cities |

---

## Section 3: The Medallion Architecture

We'll organize our data using the **medallion architecture**, a data design pattern that organizes data into layers:

```
┌─────────────────────────────────────────────────────────────────────┐
│                         MEDALLION ARCHITECTURE                       │
├─────────────────────────────────────────────────────────────────────┤
│                                                                      │
│   ┌──────────┐      ┌──────────┐      ┌──────────┐                 │
│   │  BRONZE  │─────▶│  SILVER  │─────▶│   GOLD   │                 │
│   │  (Raw)   │      │ (Cleaned)│      │(Business)│                 │
│   └──────────┘      └──────────┘      └──────────┘                 │
│                                                                      │
│   • Raw ingestion    • Quality filters  • Pre-aggregated            │
│   • Schema-on-read   • Parsed JSON      • Business metrics          │
│   • Full fidelity    • Enriched fields  • Dashboard-ready           │
│                      • Joined dims                                   │
│                                                                      │
└─────────────────────────────────────────────────────────────────────┘
```

### Why Medallion?

| Layer | Purpose | Users |
|-------|---------|-------|
| **Bronze** | Preserve raw data exactly as received | Data engineers, debugging |
| **Silver** | Single source of truth, cleaned and enriched | Data scientists, analysts |
| **Gold** | Pre-computed metrics for specific use cases | Dashboards, executives |

### Our Tables

| Layer | Table | Description |
|-------|-------|-------------|
| Bronze | `bronze.dim_*` | Dimension tables (brands, items, locations, categories) |
| Bronze | `bronze.orders` | Raw order lifecycle events |
| Silver | `silver.orders_enriched` | Cleaned events with parsed JSON and time features |
| Silver | `silver.order_lifecycle` | One row per order with all timestamps and durations |
| Gold | `gold.hourly_metrics` | Orders and revenue by hour and location |
| Gold | `gold.delivery_performance` | Delivery time stats by date and location |
| Gold | `gold.brand_summary` | Brand-level performance metrics |

---

## Section 4: Environment Setup

Now let's set up our Spark environment. In the **imperative approach**, we explicitly:

1. Create a SparkSession
2. Import required modules
3. Configure logging

**Compare with Spark 4.1:** In the declarative approach, `spark` is automatically injected by the framework.

In [ ]:
# =============================================================================
# SECTION 4: ENVIRONMENT SETUP
# =============================================================================
# 
# In the IMPERATIVE approach (Spark 4.0), we manually:
#   1. Create the SparkSession with all configuration
#   2. Import the modules we need
#   3. Set up logging preferences
#
# This gives us FULL CONTROL but requires more boilerplate code.
#
# NOTE: This notebook runs in LOCAL MODE for learning purposes.
#       For production, use the standalone scripts with spark-submit.
# =============================================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as f  # Spark's built-in functions (like SQL functions)
from pyspark.sql.types import (
    StructType,      # For defining complex schemas
    StructField,     # Individual field in a schema
    StringType,      # String data type
    IntegerType,     # Integer data type
    DoubleType,      # Floating point data type
    TimestampType,   # Timestamp data type
)

# Create SparkSession - the entry point for all Spark operations
# The builder pattern lets us configure the session before creating it
spark = SparkSession.builder \
    .appName("Spark40_Imperative_Pipeline") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

# Reduce log verbosity - we only want warnings and errors, not info messages
# This keeps our notebook output clean
spark.sparkContext.setLogLevel("WARN")

# Print version to confirm we're using Spark
print(f"Spark Version: {spark.version}")
print(f"Application Name: {spark.sparkContext.appName}")
print(f"Master: {spark.sparkContext.master}")
print("\n[OK] SparkSession created successfully")

### Understanding the Imports

| Import | Purpose |
|--------|---------||
| `SparkSession` | Main entry point - creates and manages Spark |
| `functions as f` | Built-in functions (col, sum, avg, etc.) - aliased as `f` for brevity |
| `StructType/StructField` | Define schemas for parsing JSON or creating DataFrames |
| `*Type` classes | Data type definitions for schema fields |

**Best Practice:** Always alias `functions` as `f` - it's the PySpark convention and makes code more readable.

---

## Section 5: Bronze Layer - Raw Data Ingestion

The Bronze layer is our **landing zone**. We ingest data with minimal transformation to preserve the original format.

### Imperative Approach Characteristics

In the imperative style, we:
- Explicitly define each function
- Call functions in the order we want
- Manually handle the execution flow
- Write data to tables ourselves

**Compare with Spark 4.1:** The declarative approach uses `@dp.materialized_view` decorators and automatic dependency resolution.

In [ ]:
# =============================================================================
# SECTION 5.1: LOAD DIMENSION TABLES
# =============================================================================
#
# Dimension tables are small, slowly-changing reference data.
# They describe the "what" - brands, items, locations.
#
# In IMPERATIVE style, we define functions and call them explicitly.
#
# NOTE: In local mode, we use temp views instead of Iceberg tables.
#       The transformation logic remains identical.
# =============================================================================

# Detect if running in Docker Jupyter or cluster
import os
DATA_PATH = "/home/jovyan/data" if os.path.exists("/home/jovyan/data") else "/data"

def load_dimension_table(name: str, path: str):
    """
    Load a dimension table from parquet and register as temp view.
    
    This is a REUSABLE helper function - a common pattern in imperative code.
    We define generic functions and call them with different parameters.
    
    Args:
        name: Target table name (e.g., "dim_brands")
        path: Source parquet file path
    
    Returns:
        The loaded DataFrame (also registered as temp view)
    """
    # Read parquet file - Spark automatically infers the schema
    df = spark.read.parquet(path)
    
    # Register as temp view (in production, would write to Iceberg)
    # Temp views are session-scoped and perfect for learning
    df.createOrReplaceTempView(f"bronze_{name}")
    
    # Return the DataFrame - this allows direct inspection!
    return df

# Define our dimension tables - this is our "configuration"
# In imperative code, we often separate config from logic
DIMENSION_TABLES = {
    "dim_categories": f"{DATA_PATH}/dimensions/categories.parquet",
    "dim_brands": f"{DATA_PATH}/dimensions/brands.parquet",
    "dim_items": f"{DATA_PATH}/dimensions/items.parquet",
    "dim_locations": f"{DATA_PATH}/dimensions/locations.parquet",
}

# Execute: Load each dimension table
# We MANUALLY iterate and call our function for each table
print("Loading dimension tables to Bronze layer...")
print("=" * 50)

# Store returned DataFrames for verification
loaded_dfs = {}
for table_name, file_path in DIMENSION_TABLES.items():
    df = load_dimension_table(table_name, file_path)
    loaded_dfs[table_name] = df
    print(f"  [OK] bronze_{table_name}: {df.count():,} rows")

print("\n[OK] All dimension tables loaded")

# =============================================================================
# VERIFICATION: Functions return DataFrames that you can inspect directly!
# =============================================================================
print("\n" + "=" * 50)
print("VERIFY: load_dimension_table() returns DataFrame")
print("=" * 50)

brands_df = loaded_dfs["dim_brands"]
print(f"\nType: {type(brands_df).__name__}")
print(f"Count: {brands_df.count()}")
brands_df.show(5, truncate=False)

In [ ]:
# =============================================================================
# SECTION 5.2: EXPLORE DIMENSION DATA
# =============================================================================
#
# Let's look at what we loaded to understand the data.
# =============================================================================

# Show the brands - these are our virtual restaurants
print("BRANDS (Virtual Restaurants)")
print("=" * 50)
spark.table("bronze_dim_brands").show(truncate=False)

In [ ]:
# Show the locations - these are our delivery markets
print("LOCATIONS (Delivery Markets)")
print("=" * 50)
spark.table("bronze_dim_locations").show(truncate=False)

In [ ]:
# =============================================================================
# SECTION 5.3: LOAD ORDER EVENTS (FACT TABLE)
# =============================================================================
#
# The orders table is our FACT table - the core transactional data.
# This is where most of our data volume lives.
#
# Key transformation: Parse the timestamp string to a proper timestamp type.
#
# NOTE: In local mode, we use the 1-day sample (~160K events) for speed.
#       Production uses the full 90-day dataset (~14.5M events).
# =============================================================================

def load_orders():
    """
    Load order lifecycle events from parquet and register as temp view.
    
    This function includes a MINIMAL transformation - parsing the timestamp.
    We do this in Bronze because timestamps are fundamental to all downstream work.
    
    Returns:
        The loaded DataFrame (also registered as temp view)
    """
    # Use 1-day sample for local mode (faster), 90-day for production
    orders_file = "orders_1d.parquet" if os.path.exists(f"{DATA_PATH}/events/orders_1d.parquet") else "orders_90d.parquet"
    path = f"{DATA_PATH}/events/{orders_file}"
    
    # Read the raw parquet data
    df = spark.read.parquet(path)
    
    # Parse the timestamp string to a proper TimestampType
    # The source format is "2024-01-15T14:30:00" (ISO format with T separator)
    # We replace T with space to match Spark's expected format
    #
    # f.regexp_replace: Replace pattern in string
    # f.to_timestamp: Convert string to timestamp
    df = df.withColumn(
        "event_timestamp",
        f.to_timestamp(f.regexp_replace("ts", "T", " "))
    )
    
    # Register as temp view
    df.createOrReplaceTempView("bronze_orders")
    
    # Return the DataFrame for direct inspection
    return df

# Execute: Load orders
print("Loading order events to Bronze layer...")
print("=" * 50)

orders_df = load_orders()
print(f"  [OK] bronze_orders: {orders_df.count():,} rows")
print("\n[OK] Bronze layer complete")

# =============================================================================
# VERIFICATION: load_orders() returns DataFrame with timestamp transformation
# =============================================================================
print("\n" + "=" * 50)
print("VERIFY: load_orders() returns DataFrame")
print("=" * 50)
print(f"\nType: {type(orders_df).__name__}")
print(f"Count: {orders_df.count():,}")
orders_df.select("event_id", "event_type", "order_id", "event_timestamp").show(5)

In [ ]:
# =============================================================================
# SECTION 5.4: EXPLORE ORDER DATA
# =============================================================================
#
# Let's understand the structure of our order events.
# =============================================================================

# Show the schema - what fields do we have?
print("ORDER EVENTS SCHEMA")
print("=" * 50)
spark.table("bronze_orders").printSchema()

In [ ]:
# Show sample data
print("SAMPLE ORDER EVENTS")
print("=" * 50)
spark.table("bronze_orders") \
    .select("event_id", "event_type", "order_id", "location_id", "event_timestamp", "body") \
    .show(10, truncate=40)

In [ ]:
# Count events by type - understand the event distribution
print("EVENT TYPE DISTRIBUTION")
print("=" * 50)
spark.table("bronze_orders") \
    .groupBy("event_type") \
    .count() \
    .orderBy(f.desc("count")) \
    .show()

### Section 5 Summary: Bronze Layer

**What we did:**
- Loaded 4 dimension tables (brands, categories, items, locations)
- Loaded ~14.5M order lifecycle events
- Added a parsed timestamp column

**Imperative patterns used:**
- Defined reusable helper functions
- Used loops to process multiple tables
- Called functions explicitly in order
- Manually wrote data to tables

**Compare with Spark 4.1:** The declarative version uses `@dp.materialized_view` decorators instead of explicit function definitions and calls. The framework handles execution order automatically.

---

## Section 6: Silver Layer - Data Transformation

The Silver layer is where we **clean, enrich, and transform** our data into a queryable format.

### Transformations We'll Apply

1. **Filter nulls** - Remove records with missing critical fields
2. **Parse JSON** - Extract structured data from the `body` field
3. **Add features** - Compute time-based features (hour, day of week, weekend flag)
4. **Join dimensions** - Enrich with location names
5. **Pivot lifecycle** - Create one row per order with all event timestamps

In [ ]:
# =============================================================================
# SECTION 6.1: TRANSFORM ORDERS TO SILVER - ORDERS_ENRICHED
# =============================================================================
#
# This is our MAIN transformation function. It takes raw bronze data and
# produces clean, enriched silver data.
#
# In IMPERATIVE style, we write one large function with all the logic.
# This gives us fine-grained control over each step.
# =============================================================================

def transform_orders_to_silver():
    """
    Transform bronze orders to silver layer with cleaning and enrichment.
    
    This function demonstrates the IMPERATIVE transformation pattern:
    - Read input tables explicitly
    - Apply transformations in sequence
    - Write output explicitly
    
    Returns:
        The enriched DataFrame (also registered as temp view)
    """
    
    # =========================================================================
    # STEP 1: Read source data
    # =========================================================================
    # We explicitly read the tables we need
    orders = spark.table("bronze_orders")
    print(f"  Read bronze_orders: {orders.count():,} rows")
    
    # =========================================================================
    # STEP 2: Filter null values
    # =========================================================================
    # Remove records with missing critical fields
    # The & operator combines boolean conditions (AND)
    # isNotNull() returns True if the column has a value
    cleaned = orders.filter(
        f.col("event_id").isNotNull() &
        f.col("order_id").isNotNull() &
        f.col("event_timestamp").isNotNull()
    )
    print(f"  After null filter: {cleaned.count():,} rows")
    
    # =========================================================================
    # STEP 3: Parse JSON body
    # =========================================================================
    # The 'body' field contains JSON with event-specific data.
    # We define a schema and use from_json() to parse it.
    #
    # StructType defines the expected JSON structure
    # StructField defines each field: (name, type, nullable)
    body_schema = StructType([
        StructField("brand_id", IntegerType(), True),
        StructField("item_ids", StringType(), True),
        StructField("total", DoubleType(), True),
        StructField("lat", DoubleType(), True),
        StructField("lng", DoubleType(), True),
        StructField("driver_id", StringType(), True),
    ])
    
    # Parse JSON and add as new column 'body_parsed'
    enriched = cleaned.withColumn(
        "body_parsed",
        f.from_json(f.col("body"), body_schema)
    )
    
    # =========================================================================
    # STEP 4: Extract parsed fields
    # =========================================================================
    # Use select() to choose and rename columns
    # body_parsed.field_name accesses nested struct fields
    # alias() renames a column in the output
    enriched = enriched.select(
        # Keep original columns
        "event_id",
        "event_type",
        "event_timestamp",
        "ts",
        "ts_seconds",
        "order_id",
        "location_id",
        "sequence",
        "body",  # Keep raw body for debugging
        # Extract parsed fields with descriptive names
        f.col("body_parsed.brand_id").alias("brand_id"),
        f.col("body_parsed.total").alias("order_total"),
        f.col("body_parsed.lat").alias("latitude"),
        f.col("body_parsed.lng").alias("longitude"),
        f.col("body_parsed.driver_id").alias("driver_id"),
    )
    
    # =========================================================================
    # STEP 5: Add time-based features
    # =========================================================================
    # withColumns() adds multiple columns at once (Spark 3.3+)
    # These features enable time-based analysis:
    #   - event_hour: For hourly patterns (lunch rush, dinner peak)
    #   - event_day_of_week: For weekly patterns (weekday vs weekend)
    #   - is_weekend: Boolean flag for weekend filtering
    #   - event_date: Date only (for daily aggregations)
    enriched = enriched.withColumns({
        "event_hour": f.hour("event_timestamp"),
        "event_day_of_week": f.dayofweek("event_timestamp"),
        "is_weekend": f.when(
            f.dayofweek("event_timestamp").isin(1, 7),  # 1=Sunday, 7=Saturday in Spark
            True
        ).otherwise(False),
        "event_date": f.to_date("event_timestamp"),
    })
    
    # =========================================================================
    # STEP 6: Join with locations dimension
    # =========================================================================
    # Read locations and select only the fields we need
    # Using alias() to rename 'id' to 'location_id' for the join
    locations = spark.table("bronze_dim_locations").select(
        f.col("id").alias("location_id"),
        f.col("city").alias("city_name"),
    )
    
    # broadcast() hint tells Spark to send the small table to all workers
    # This is much faster than shuffling the large orders table
    # Use broadcast for small dimension tables (< few MB)
    enriched = enriched.join(
        f.broadcast(locations),
        on="location_id",
        how="left"  # Keep all orders even if location missing
    )
    
    # =========================================================================
    # STEP 7: Register as Silver temp view
    # =========================================================================
    enriched.createOrReplaceTempView("silver_orders_enriched")
    
    count = enriched.count()
    print(f"  [OK] silver_orders_enriched: {count:,} rows")
    
    # Return the DataFrame for direct inspection
    return enriched

# Execute the transformation
print("Transforming orders to Silver layer...")
print("=" * 50)
enriched_df = transform_orders_to_silver()

# =============================================================================
# VERIFICATION: transform_orders_to_silver() returns DataFrame
# =============================================================================
print("\n" + "=" * 50)
print("VERIFY: transform_orders_to_silver() returns DataFrame")
print("=" * 50)
print(f"\nType: {type(enriched_df).__name__}")
print(f"Count: {enriched_df.count():,}")
enriched_df.filter(f.col("event_type") == "order_created") \
    .select("order_id", "city_name", "brand_id", "order_total", "event_hour", "is_weekend") \
    .show(5)

In [ ]:
# =============================================================================
# SECTION 6.2: EXPLORE ENRICHED DATA
# =============================================================================

# Show the new schema with all our added fields
print("ENRICHED ORDERS SCHEMA")
print("=" * 50)
spark.table("silver_orders_enriched").printSchema()

In [ ]:
# Show sample enriched data
print("SAMPLE ENRICHED ORDERS")
print("=" * 50)
spark.table("silver_orders_enriched") \
    .select(
        "event_type", "order_id", "city_name", "brand_id",
        "order_total", "event_hour", "is_weekend"
    ) \
    .filter(f.col("event_type") == "order_created") \
    .show(10)

In [ ]:
# =============================================================================
# SECTION 6.3: CREATE ORDER LIFECYCLE TABLE
# =============================================================================
#
# This transformation PIVOTS the event data so we get one row per order
# with timestamps for each lifecycle stage.
#
# Input:  Multiple rows per order (one per event)
# Output: One row per order with columns for each event timestamp
# =============================================================================

def create_order_lifecycle():
    """
    Create a pivoted view of order lifecycle with timestamps for each stage.
    
    This enables analysis of:
    - Kitchen preparation time
    - Delivery duration
    - Total order-to-delivery time
    
    Returns:
        The lifecycle DataFrame (completed orders only, also registered as temp view)
    """
    
    # Read enriched orders
    orders = spark.table("silver_orders_enriched")
    
    # =========================================================================
    # STEP 1: Pivot events to columns
    # =========================================================================
    # pivot() transforms rows into columns:
    #   - groupBy columns become the row identifiers
    #   - pivot column values become new column names
    #   - agg() specifies what value to put in each cell
    #
    # We use min() because an order might have duplicate events (data quality)
    # and we want the earliest timestamp for each event type.
    lifecycle = orders.groupBy(
        "order_id",
        "location_id",
        "city_name"
    ).pivot(
        "event_type",
        # Specify exact values to control column order and avoid scanning data
        ["order_created", "kitchen_started", "kitchen_finished", "order_ready",
         "driver_arrived", "driver_picked_up", "delivered"]
    ).agg(
        f.min("event_timestamp").alias("ts")
    )
    
    # =========================================================================
    # STEP 2: Rename columns for clarity
    # =========================================================================
    # The pivot creates columns named after event_type values.
    # We rename them to be more descriptive.
    lifecycle = lifecycle.select(
        "order_id",
        "location_id",
        "city_name",
        f.col("order_created").alias("created_at"),
        f.col("kitchen_started").alias("kitchen_started_at"),
        f.col("kitchen_finished").alias("kitchen_finished_at"),
        f.col("order_ready").alias("order_ready_at"),
        f.col("driver_arrived").alias("driver_arrived_at"),
        f.col("driver_picked_up").alias("pickup_at"),
        f.col("delivered").alias("delivered_at"),
    )
    
    # =========================================================================
    # STEP 3: Calculate duration metrics
    # =========================================================================
    # unix_timestamp() converts timestamp to seconds since epoch
    # Subtracting gives duration in seconds; divide by 60 for minutes
    lifecycle = lifecycle.withColumns({
        # Kitchen time: from kitchen_started to kitchen_finished
        "kitchen_duration_min": (
            f.unix_timestamp("kitchen_finished_at") -
            f.unix_timestamp("kitchen_started_at")
        ) / 60,
        
        # Delivery time: from pickup to delivered
        "delivery_duration_min": (
            f.unix_timestamp("delivered_at") -
            f.unix_timestamp("pickup_at")
        ) / 60,
        
        # Total time: from order created to delivered
        "total_duration_min": (
            f.unix_timestamp("delivered_at") -
            f.unix_timestamp("created_at")
        ) / 60,
    })
    
    # =========================================================================
    # STEP 4: Filter to completed orders only
    # =========================================================================
    # Only include orders that have been delivered
    # This excludes cancelled orders and orders still in progress
    completed = lifecycle.filter(f.col("delivered_at").isNotNull())
    
    # Register as temp view
    completed.createOrReplaceTempView("silver_order_lifecycle")
    
    count = completed.count()
    print(f"  [OK] silver_order_lifecycle: {count:,} rows (completed orders)")
    
    # Return the DataFrame for direct inspection
    return completed

# Execute
print("\nCreating order lifecycle table...")
print("=" * 50)
lifecycle_df = create_order_lifecycle()
print("\n[OK] Silver layer complete")

# =============================================================================
# VERIFICATION: create_order_lifecycle() returns DataFrame
# =============================================================================
print("\n" + "=" * 50)
print("VERIFY: create_order_lifecycle() returns DataFrame")
print("=" * 50)
print(f"\nType: {type(lifecycle_df).__name__}")
print(f"Count: {lifecycle_df.count():,}")
lifecycle_df.select(
    "order_id", "city_name",
    f.round("kitchen_duration_min", 1).alias("kitchen_min"),
    f.round("delivery_duration_min", 1).alias("delivery_min"),
    f.round("total_duration_min", 1).alias("total_min")
).show(5)

In [ ]:
# =============================================================================
# SECTION 6.4: EXPLORE LIFECYCLE DATA
# =============================================================================

print("SAMPLE ORDER LIFECYCLE")
print("=" * 50)
spark.table("silver_order_lifecycle") \
    .select(
        "order_id", "city_name",
        f.round("kitchen_duration_min", 1).alias("kitchen_min"),
        f.round("delivery_duration_min", 1).alias("delivery_min"),
        f.round("total_duration_min", 1).alias("total_min")
    ) \
    .show(10)

### Section 6 Summary: Silver Layer

**What we did:**
- Cleaned data by filtering null values
- Parsed JSON body field to extract structured data
- Added time-based features (hour, day of week, weekend flag)
- Joined with location dimension for city names
- Created pivoted lifecycle table with duration metrics

**Imperative patterns used:**
- Long functions with step-by-step transformations
- Explicit print statements for progress tracking
- Manual control over transformation order

**Compare with Spark 4.1:** In the declarative version, each table is defined with a `@dp.materialized_view` decorator. Dependencies between tables (like `orders_enriched` → `order_lifecycle`) are inferred automatically from `spark.table()` calls.

---

## Section 7: Gold Layer - Business Aggregations

The Gold layer contains **pre-aggregated metrics** optimized for specific business questions. These tables power dashboards and reports.

### Why Pre-Aggregate?

| Approach | Query Time | Storage | Flexibility |
|----------|-----------|---------|-------------|
| Query Silver directly | Slow (scans millions of rows) | None | High |
| Pre-aggregate to Gold | Fast (scans thousands of rows) | Some | Medium |

**Rule of thumb:** Pre-aggregate when the same aggregation is queried frequently.

In [ ]:
# =============================================================================
# SECTION 7.1: HOURLY METRICS
# =============================================================================
#
# Business Question: What are the order volumes and revenue by hour and location?
#
# This powers:
#   - Real-time operations dashboard
#   - Staffing decisions (when do we need more kitchen staff?)
#   - Marketing (when should we run promotions?)
# =============================================================================

def create_hourly_metrics():
    """
    Create hourly order metrics by location.
    
    Aggregations:
    - order_count: Number of orders placed
    - total_revenue: Sum of order totals
    - avg_order_value: Average order size
    - unique_brands: Number of distinct brands ordered from
    
    Returns:
        The aggregated DataFrame (also registered as temp view)
    """
    orders = spark.table("silver_orders_enriched")
    
    # Filter to order_created events only - one per order
    # This avoids double-counting orders
    hourly = orders.filter(
        f.col("event_type") == "order_created"
    ).groupBy(
        # Group by dimensions we want to analyze
        "event_date",
        "event_hour",
        "location_id",
        "city_name",
    ).agg(
        # Aggregate functions
        f.count("order_id").alias("order_count"),
        f.sum("order_total").alias("total_revenue"),
        f.avg("order_total").alias("avg_order_value"),
        f.countDistinct("brand_id").alias("unique_brands"),
    )
    
    hourly.createOrReplaceTempView("gold_hourly_metrics")
    
    count = hourly.count()
    print(f"  [OK] gold_hourly_metrics: {count:,} rows")
    
    # Return the DataFrame for direct inspection
    return hourly

# Execute
print("Creating Gold layer aggregations...")
print("=" * 50)
hourly_df = create_hourly_metrics()

# =============================================================================
# VERIFICATION: create_hourly_metrics() returns DataFrame
# =============================================================================
print(f"\n  VERIFY: Type={type(hourly_df).__name__}, Count={hourly_df.count():,}")
hourly_df.orderBy(f.desc("order_count")).show(5)

In [ ]:
# =============================================================================
# SECTION 7.2: DELIVERY PERFORMANCE
# =============================================================================
#
# Business Question: How fast are we fulfilling orders? Where are the bottlenecks?
#
# This powers:
#   - Operations review meetings
#   - SLA monitoring (are we meeting delivery promises?)
#   - Kitchen optimization
# =============================================================================

def create_delivery_performance():
    """
    Create delivery performance metrics by date and location.
    
    Key metrics:
    - avg_kitchen_time_min: Average time to prepare food
    - avg_delivery_time_min: Average time from pickup to delivery
    - avg_total_time_min: Average end-to-end time
    - median_total_time_min: Median (less sensitive to outliers)
    - p95_total_time_min: 95th percentile (worst 5% of orders)
    
    Returns:
        The aggregated DataFrame (also registered as temp view)
    """
    lifecycle = spark.table("silver_order_lifecycle")
    
    performance = lifecycle.groupBy(
        f.to_date("created_at").alias("order_date"),
        "location_id",
        "city_name",
    ).agg(
        f.count("order_id").alias("completed_orders"),
        
        # Average times
        f.avg("kitchen_duration_min").alias("avg_kitchen_time_min"),
        f.avg("delivery_duration_min").alias("avg_delivery_time_min"),
        f.avg("total_duration_min").alias("avg_total_time_min"),
        
        # Percentiles for better understanding of distribution
        # percentile_approx is faster than exact percentile for large datasets
        f.percentile_approx("total_duration_min", 0.5).alias("median_total_time_min"),
        f.percentile_approx("total_duration_min", 0.95).alias("p95_total_time_min"),
    )
    
    performance.createOrReplaceTempView("gold_delivery_performance")
    
    count = performance.count()
    print(f"  [OK] gold_delivery_performance: {count:,} rows")
    
    # Return the DataFrame for direct inspection
    return performance

delivery_df = create_delivery_performance()

# =============================================================================
# VERIFICATION: create_delivery_performance() returns DataFrame
# =============================================================================
print(f"\n  VERIFY: Type={type(delivery_df).__name__}, Count={delivery_df.count():,}")
delivery_df.select(
    "city_name", "completed_orders",
    f.round("avg_total_time_min", 1).alias("avg_min"),
    f.round("p95_total_time_min", 1).alias("p95_min")
).show(5)

In [ ]:
# =============================================================================
# SECTION 7.3: BRAND SUMMARY
# =============================================================================
#
# Business Question: How are our virtual brands performing?
#
# This powers:
#   - Brand portfolio decisions (which brands to grow/sunset)
#   - Marketing budget allocation
#   - Menu optimization
# =============================================================================

def create_brand_summary():
    """
    Create brand-level summary metrics.
    
    Joins order data with brand dimension for brand names.
    
    Returns:
        The aggregated DataFrame (one per brand, also registered as temp view)
    """
    orders = spark.table("silver_orders_enriched")
    brands = spark.table("bronze_dim_brands")
    
    # Aggregate by brand
    brand_metrics = orders.filter(
        f.col("event_type") == "order_created"
    ).groupBy("brand_id").agg(
        f.count("order_id").alias("total_orders"),
        f.sum("order_total").alias("total_revenue"),
        f.avg("order_total").alias("avg_order_value"),
        f.countDistinct("location_id").alias("locations_served"),
        f.min("event_date").alias("first_order_date"),
        f.max("event_date").alias("last_order_date"),
    )
    
    # Join with brand names
    summary = brand_metrics.join(
        brands.select(f.col("id").alias("brand_id"), "name"),
        on="brand_id",
        how="left"
    ).select(
        "brand_id",
        f.col("name").alias("brand_name"),
        "total_orders",
        "total_revenue",
        "avg_order_value",
        "locations_served",
        "first_order_date",
        "last_order_date",
    )
    
    summary.createOrReplaceTempView("gold_brand_summary")
    
    count = summary.count()
    print(f"  [OK] gold_brand_summary: {count:,} rows")
    
    # Return the DataFrame for direct inspection
    return summary

brand_df = create_brand_summary()
print("\n[OK] Gold layer complete")

# =============================================================================
# VERIFICATION: create_brand_summary() returns DataFrame
# =============================================================================
print("\n" + "=" * 50)
print("VERIFY: create_brand_summary() returns DataFrame")
print("=" * 50)
print(f"\nType: {type(brand_df).__name__}")
print(f"Count: {brand_df.count()}")
brand_df.select(
    "brand_name", "total_orders",
    f.round("total_revenue", 2).alias("revenue"),
    f.round("avg_order_value", 2).alias("aov")
).orderBy(f.desc("total_revenue")).show(5)

In [ ]:
# =============================================================================
# SECTION 7.4: EXPLORE GOLD DATA
# =============================================================================

print("TOP PERFORMING BRANDS")
print("=" * 50)
spark.table("gold_brand_summary") \
    .select(
        "brand_name",
        "total_orders",
        f.round("total_revenue", 2).alias("total_revenue"),
        f.round("avg_order_value", 2).alias("avg_order_value")
    ) \
    .orderBy(f.desc("total_revenue")) \
    .show(10)

In [ ]:
print("DELIVERY PERFORMANCE BY CITY")
print("=" * 50)
spark.table("gold_delivery_performance") \
    .groupBy("city_name") \
    .agg(
        f.sum("completed_orders").alias("total_orders"),
        f.round(f.avg("avg_total_time_min"), 1).alias("avg_time_min"),
        f.round(f.avg("p95_total_time_min"), 1).alias("p95_time_min")
    ) \
    .orderBy(f.desc("total_orders")) \
    .show()

---

## Section 8: Pipeline Summary

### What We Built

```
DATA SOURCES                     BRONZE                  SILVER                     GOLD
-----------------------------------------------------------------------------------------------
                                                                                 
/data/dimensions/*.parquet ----> bronze_dim_brands                                       
                                 bronze_dim_categories                                    
                                 bronze_dim_items                                         
                                 bronze_dim_locations ------+                          
                                                            |                          
/data/events/orders.parquet ---> bronze_orders -------------+---> silver_orders_enriched ----> gold_hourly_metrics
                                                                  |                            gold_brand_summary
                                                                  |                          
                                                                  +---> silver_order_lifecycle --> gold_delivery_performance
```

### Tables Created (as Temp Views)

| Layer | Table | Purpose |
|-------|-------|---------|
| Bronze | bronze_dim_brands | Virtual restaurant brands |
| Bronze | bronze_dim_categories | Food categories |
| Bronze | bronze_dim_items | Menu items |
| Bronze | bronze_dim_locations | Delivery markets |
| Bronze | bronze_orders | Raw order lifecycle events |
| Silver | silver_orders_enriched | Cleaned and enriched events |
| Silver | silver_order_lifecycle | One row per completed order |
| Gold | gold_hourly_metrics | Hourly aggregations |
| Gold | gold_delivery_performance | Daily delivery stats |
| Gold | gold_brand_summary | Brand-level metrics |

**Note:** In local mode, we use temp views. In production with the full Lakehouse Stack, these would be Iceberg tables for durability and time-travel.

---

## Section 9: Imperative Approach - Tradeoffs

### Advantages of the Imperative Approach

| Advantage | Why It Matters |
|-----------|----------------|
| **Full control** | You decide exactly when and how each step runs |
| **Easy debugging** | Add print statements anywhere, step through code |
| **Familiar pattern** | Standard Python functions - no new concepts |
| **Flexible execution** | Run individual functions, skip steps, re-run specific parts |
| **No framework lock-in** | Pure PySpark code works on any Spark distribution |

### Disadvantages of the Imperative Approach

| Disadvantage | Impact |
|--------------|--------|
| **Manual dependency management** | You must call functions in correct order |
| **No automatic parallelism** | You must manually parallelize independent operations |
| **Verbose boilerplate** | Lots of explicit read/write/function calls |
| **Error handling is manual** | You must add try/except blocks yourself |
| **No built-in validation** | You must write your own data quality checks |

### When to Use Imperative Style

The imperative approach is a good fit when:

- You need maximum flexibility and control
- You're exploring data interactively in notebooks
- Your pipeline logic is complex and non-linear
- You want to avoid framework dependencies
- You need to debug step-by-step

### Compare with Spark 4.1 Declarative

Open the `spark41_declarative_pipeline.ipynb` notebook to see how the same pipeline is expressed using Spark Declarative Pipelines. Key differences:

| Aspect | Imperative (4.0) | Declarative (4.1) |
|--------|------------------|-------------------|
| Execution order | Manual (you call functions) | Automatic (framework resolves) |
| Dependencies | Implicit (in your call order) | Explicit (from spark.table calls) |
| Boilerplate | More (read/write in each function) | Less (decorators + return) |
| Validation | Manual | Built-in dry-run |
| CLI integration | None | spark-pipelines command |

In [ ]:
# =============================================================================
# CLEANUP
# =============================================================================

# Stop the Spark session to release resources
# In production, you'd typically let the cluster manage this
# spark.stop()

print("\n" + "=" * 50)
print("WORKSHOP COMPLETE")
print("=" * 50)
print("\nYou've built a complete data pipeline using the imperative approach.")
print("\nNext steps:")
print("  1. Open spark41_declarative_pipeline.ipynb to compare approaches")
print("  2. Query the Gold tables to answer business questions")
print("  3. Try the production scripts with spark-submit on the full cluster")